# 论文复现：三维卷积神经网络用于机载高光谱图像树种分类

复现对象：Zhang B., Zhao L., Zhang X. (2020). *Three-dimensional convolutional neural network model for tree species classification using airborne hyperspectral images*. Remote Sensing of Environment, 247, 111938.

本 notebook 将论文的两个模型（**3D-CNN** 与 **3D-1D-CNN**）及其改进版（BatchNorm + 全局平均池化分类头）的训练与评测流程整合为一段自包含流程，便于他人克隆仓库后直接复现。

**如何使用**：只需修改下方「配置区」中的 `DATASET_NAME` 与 `MODEL_NAME` 两个变量，即可切换数据集 / 模型，其余流程（预处理、划分、训练、评测）保持一致。

**统一口径**（三个数据集一致）：

- 原始光谱波段（`reducer=none`，不做 PCA）
- 空间 patch 固定 11×11
- 逐波段 z-score 标准化（仅训练集拟合）
- 固定划分 `fair24_6_70`（24% 训练 / 6% 验证 / 70% 测试）+ `seed=1442`


## 0. 配置区（改这里即可切换数据集 / 模型）

In [ ]:
# =====================================================================
# 只需修改下面两个变量，即可切换数据集 / 模型，其余流程无需改动
# =====================================================================
DATASET_NAME = "pavia_university"   # 可选: "pavia_university" | "indian_pines" | "salinas"
MODEL_NAME   = "paper3dcnn"     # 可选: "paper3dcnn" | "paper3d1dcnn" | "improvedpaper3d1dcnn"

# ---- 训练协议（论文设定，一般无需改动）----
SEED = 1442
SPLIT_PROTOCOL = "fair24_6_70"      # 24% 训练 / 6% 验证 / 70% 测试，固定 seed
BATCH_SIZE = 64
EPOCHS = 300
LEARNING_RATE = 1e-4
MOMENTUM = 0.9
DROPOUT = 0.5
PATCH_SIZE = 11                     # 论文空间 patch 固定 11×11（严格复现）

# 早停：论文模型按验证 loss，Δ < MIN_DELTA 且连续 PATIENCE 轮不改善则停
EARLY_STOP_METRIC = "loss"
EARLY_STOP_MIN_DELTA = 0.003
EARLY_STOP_PATIENCE = 10


## 1. 环境与导入

In [ ]:
import sys
from pathlib import Path

# 定位项目根（实验交付/），并加入 sys.path，保证从任意目录运行均可导入 src
PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").is_file():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn

from src.datasets.高光谱预处理 import (
    PreprocessingConfig,
    load_hsi_data,
    HSIPreprocessingPipeline,
)
from src.evaluation.classification_metrics import classification_summary
from src.models.Paper3D1DCNN import Paper3DCNN, Paper3D1DCNN, count_trainable_parameters
from src.models.改进Paper3D1DCNN import ImprovedPaper3D1DCNN
from src.utils.reproducibility import seed_everything

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"项目根: {PROJECT_ROOT}")
print(f"设备: {DEVICE}   torch={torch.__version__}   numpy={np.__version__}")


## 2. 模型（论文 3D-CNN / 3D-1D-CNN / 改进版）

三个模型共用「五层有效 3D 卷积」主干，输入 `[N, 1, B, 11, 11]`（B 为原始波段数，光谱维放在第 3 维）：

| 模型 | 结构要点 | 论文参数量 (B=125, C=12) |
|---|---|---|
| **Paper3DCNN** | 5×Conv3d(1→4→8→16→32→64, k=(7,3,3), valid) → Dropout → Flatten(64·(B−30)) → FC(128) → FC(C) | **951,652** |
| **Paper3D1DCNN** | 同一 3D 主干 → 重排 [N,64,B−30] → 2×Conv1d(64→48→24, k=7) → Flatten(24·(B−42)) → FC(C) | **225,292** |
| **改进Paper3D1DCNN** | 3D/1D 层加 BatchNorm + 全局平均池化分类头（无残差） | ≈20 万（随数据集 B/C 变化） |

模型实现位于 `src/models/Paper3D1DCNN.py` 与 `src/models/改进Paper3D1DCNN.py`（有单元测试校验参数量对齐）。下方 `build_model` 按 `MODEL_NAME` 分派。


In [ ]:
def build_model(name, *, spectral_bands, num_classes):
    """按名称构建论文模型，供训练分派使用。"""
    key = name.lower().replace("-", "").replace("_", "")
    if key in {"paper3dcnn", "3dcnn"}:
        return Paper3DCNN(spectral_bands, num_classes, dropout=DROPOUT)
    if key in {"paper3d1dcnn", "3d1dcnn"}:
        return Paper3D1DCNN(spectral_bands, num_classes, dropout=DROPOUT)
    if key in {"improvedpaper3d1dcnn", "paper3d1dcnnimproved"}:
        return ImprovedPaper3D1DCNN(spectral_bands, num_classes, dropout=DROPOUT)
    raise ValueError(f"未知模型名: {name!r}")


## 3. 数据加载与预处理（reducer=none + patch11，各数据集口径一致）

统一配置 `PreprocessingConfig`：`reducer=none`（保留原始波段）、`patch_size=11`、`standardization=standard`（仅训练集拟合的逐波段 z-score）。`load_hsi_data` 读取原始立方体与固定划分，`HSIPreprocessingPipeline.fit` 在训练中心像元上拟合标准化，再对全立方体做变换，最后 `build_torch_loaders` 产出 train/validation/test 三个 DataLoader。


In [ ]:
config = PreprocessingConfig(
    dataset_name=DATASET_NAME,
    split_protocol=SPLIT_PROTOCOL,
    split_seed=SEED,
    standardization="standard",
    reducer="none",              # 关键：论文模型不使用 PCA，保留原始波段
    n_components=None,           # reducer=none 时须为 None
    representation="patch",
    patch_size=PATCH_SIZE,
    padding_mode="constant",
    padding_value=0.0,
    output_dtype="float32",
)
config.validate()

data = load_hsi_data(PROJECT_ROOT, config)
pipeline = HSIPreprocessingPipeline(config).fit(data)

loaders = pipeline.build_torch_loaders(
    data, batch_size=BATCH_SIZE, loader_seed=SEED, num_workers=0,
)

num_classes = len(data.spec.class_names)
spectral_bands = pipeline.output_bands      # reducer=none → 原始有效波段数

print(f"数据集: {data.spec.name}")
print(f"类别数: {num_classes}   类别: {list(data.spec.class_names)}")
print(f"输入: [N, 1, {spectral_bands}, {PATCH_SIZE}, {PATCH_SIZE}]")
for name in ("train", "validation", "test"):
    loader = loaders.get(name)
    print(f"  {name:<10s}: {0 if loader is None else len(loader.dataset)} 个样本")


## 4. 训练（SGD + 验证 loss 早停 + 实时进度条）

优化器为 SGD（lr=1e-4，momentum=0.9），损失为交叉熵。每个 epoch 用 tqdm 实时显示 batch 级 loss/acc；每轮在验证集上评估，按验证 loss 早停（Δ<0.003，patience 10），并始终保存验证准确率最高的 checkpoint。


In [ ]:
from tqdm import tqdm

seed_everything(SEED)

# 构建模型 / 优化器 / 损失
model = build_model(MODEL_NAME, spectral_bands=spectral_bands, num_classes=num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

n_params = count_trainable_parameters(model)
print(f"模型: {MODEL_NAME}   可训练参数: {n_params:,}")


@torch.inference_mode()
def evaluate(model, loader):
    """返回 (平均 loss, 准确率, 真实标签, 预测标签)，标签均为 0 基。"""
    model.eval()
    loss_sum = 0.0
    labels_all, preds_all = [], []
    for batch in loader:
        x = batch["input"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        logits = model(x)
        loss_sum += float(criterion(logits, y)) * y.size(0)
        labels_all.append(y.cpu())
        preds_all.append(logits.argmax(dim=1).cpu())
    labels = torch.cat(labels_all).numpy()
    preds = torch.cat(preds_all).numpy()
    acc = float((labels == preds).mean())
    return loss_sum / labels.size, acc, labels, preds


# ---- 训练循环：tqdm 进度 + 验证 loss 早停 + 按验证准确率保存最优 checkpoint ----
history = []
best_val_acc = -1.0
best_state = None
best_monitor = float("inf") if EARLY_STOP_METRIC == "loss" else -float("inf")
early_stop_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(loaders["train"], desc=f"训练 epoch {epoch}/{EPOCHS}", unit="batch", leave=False)
    loss_sum, correct, total = 0.0, 0, 0
    for batch in pbar:
        x = batch["input"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        n = y.size(0)
        loss_sum += float(loss.detach()) * n
        correct += int((logits.argmax(dim=1) == y).sum())
        total += n
        pbar.set_postfix(loss=f"{float(loss.detach()):.4f}", acc=f"{correct / total:.3f}")
    pbar.close()

    train_loss = loss_sum / total
    train_acc = correct / total
    record = {"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc}

    if loaders["validation"] is not None:
        val_loss, val_acc, _, _ = evaluate(model, loaders["validation"])
        record.update({"val_loss": val_loss, "val_acc": val_acc})
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        monitor = val_loss if EARLY_STOP_METRIC == "loss" else val_acc
        if EARLY_STOP_METRIC == "loss":
            improved = monitor < best_monitor - EARLY_STOP_MIN_DELTA
        else:
            improved = monitor > best_monitor + EARLY_STOP_MIN_DELTA
        if improved:
            best_monitor = monitor
            early_stop_counter = 0
        else:
            early_stop_counter += 1

    history.append(record)
    msg = f"epoch={epoch:03d}/{EPOCHS}  train_loss={train_loss:.6f}  train_acc={train_acc:.4f}"
    if "val_acc" in record:
        msg += f"  val_loss={val_loss:.6f}  val_acc={val_acc:.4f}"
    print(msg)

    if loaders["validation"] is not None and early_stop_counter >= EARLY_STOP_PATIENCE:
        print(f"早停触发：验证 {EARLY_STOP_METRIC} 连续 {EARLY_STOP_PATIENCE} 轮无改善，停止于 epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"已恢复验证准确率最高的 checkpoint（best val_acc={best_val_acc:.4f}）")


## 5. 测试集评测（只评测一次）

训练全部结束后，在**最优 checkpoint** 上对测试集评测一次（测试集不参与早停与超参选择），输出 OA / AA / Kappa / 逐类准确率与混淆矩阵。


In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(model, loaders["test"])
summary = classification_summary(y_true, y_pred, num_classes=num_classes)
metrics = summary.to_dict(data.spec.class_names)

print("=" * 66)
print(f"数据集 {DATASET_NAME} / 模型 {MODEL_NAME}  —  测试集（只评测一次）")
print("=" * 66)
print(f"OA   (Overall Accuracy) : {metrics['oa']:.4%}")
print(f"AA   (Average Accuracy) : {metrics['aa']:.4%}")
print(f"Kappa (Cohen's)         : {metrics['kappa']:.6f}")
print(f"测试样本数              : {y_true.size}")
print("-" * 66)
print("逐类准确率：")
for row in metrics["per_class"]:
    print(f"  {row['class_name']:<28s}  支持={row['support']:>6d}  准确率={row['accuracy']:7.2%}")


## 6. 可视化（学习曲线 + 混淆矩阵）

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

epochs_list = [r["epoch"] for r in history]
train_loss = [r["train_loss"] for r in history]
train_acc = [r["train_acc"] for r in history]
val_epochs = [r["epoch"] for r in history if "val_loss" in r]
val_loss = [r["val_loss"] for r in history if "val_loss" in r]
val_acc = [r["val_acc"] for r in history if "val_acc" in r]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(epochs_list, train_loss, label="train loss")
axes[0].plot(val_epochs, val_loss, label="val loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss 曲线"); axes[0].legend()
axes[1].plot(epochs_list, train_acc, label="train acc")
axes[1].plot(val_epochs, val_acc, label="val acc")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("准确率曲线"); axes[1].legend()
plt.tight_layout()
plt.show()

cm = np.asarray(summary.confusion_matrix)
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("预测类别"); ax.set_ylabel("真实类别")
ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
ax.set_xticklabels(data.spec.class_names, rotation=90, fontsize=8)
ax.set_yticklabels(data.spec.class_names, fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title(f"{DATASET_NAME} / {MODEL_NAME} 混淆矩阵")
plt.tight_layout()
plt.show()


## 7. 结果汇总说明

- 论文模型参数量对齐测试：`Paper3DCNN=951,652`、`Paper3D1DCNN=225,292`（@ B=125, C=12，见 `tests/test_paper_models.py`）。
- 三个数据集完整结果（5 模型 × 3 数据集，`fair24_6_70` + `seed=1442`）见 `report/论文复现结果/结果分析.md` 与 `report/对比结果/对比结果.md`。
- 复现协议细节见 `论文复现_内容索引.md`。
